# Exploratory Data Analysis — Order Value Prediction

**Tasks 2.1–2.3** | Business Analytics Group Project

This notebook analyzes the merged Olist + synthetic behavioral dataset to understand order value distributions, identify key drivers, and check for multicollinearity before feature engineering.

**Data sources:**
- Olist Brazilian E-Commerce (99,441 orders, 96,096 unique customers)
- Synthetic: customer_profile (96K rows), sessions (99K rows), session_activity (764K rows)


In [ ]:
import sys

sys.path.insert(0, "..")
from src.data.eda import run_eda
from pathlib import Path

insights = run_eda(Path("plots"))

## 2.1 Target Distribution & Outliers

The order value distribution is **heavily right-skewed** (skewness = 9.37), with a median of R$105 and mean of R$160. This confirms the need for a **log transform** on the target variable before modeling with linear regressors.

**Outlier strategy:** 4.0% of delivered orders exceed the 3×IQR fence (R$519). These will be capped (winsorized) during cleaning rather than removed, to preserve high-value signal.

![Target Distribution](plots/01_target_distribution.png)


## 2.2 Order Value Drivers

### Product Category
The top-value category is **computers** (median R$1,251), followed by electronics and furniture. Low-value categories include office supplies and personal care.

![Value by Category](plots/02_value_by_category.png)

### Device Type
Desktop users have a **7% higher median order value** (R$110) than mobile (R$103), consistent with the synthetic data design where desktop correlates with deliberate, higher-value shopping sessions.

![Value by Device](plots/03_value_by_device.png)

### Traffic Source
**Email traffic** drives the highest median order value (R$146), reflecting loyal repeat customers. **Google organic** has the lowest (R$89), consistent with acquisition-stage traffic.

![Value by Traffic](plots/04_value_by_traffic.png)

### Loyalty Tier
Clear monotonic gradient: bronze (R$82) → silver (R$114) → gold (R$158) → platinum (R$202). A **2.5× spread** between lowest and highest tier.

![Value by Loyalty](plots/05_value_by_loyalty.png)

### Income Quartile
Higher income quartiles show progressively higher order values, validating the synthetic income-order value correlation (r=0.51).

![Value by Income](plots/06_value_by_income_quartile.png)

### Behavioral Features
Session duration, cart quantity, and product views all show positive relationships with order value, confirming the value-conditioned synthetic data design.

![Scatter Drivers](plots/07_scatter_drivers.png)
![Session Behavior](plots/08_session_behavior_vs_value.png)


## 2.3 Correlations & Multicollinearity

### Correlation with Target (Order Value)

| Feature | r | Type |
|---|---|---|
| avg_item_price | 0.921 | Real (⚠️ leakage risk) |
| total_cart_qty | 0.574 | Synthetic |
| monthly_income | 0.513 | Synthetic |
| freight_value | 0.491 | Real |
| n_product_views | 0.485 | Synthetic |
| n_events | 0.460 | Synthetic |
| session_duration_min | 0.357 | Synthetic |
| household_size | 0.193 | Synthetic |
| item_count | 0.190 | Real |

**Key insight:** Synthetic behavioral features carry **genuine predictive signal** (r = 0.19–0.57), validating the value-conditioned generation design. Without conditioning, these correlations would be ≈ 0.

**⚠️ Leakage warning:**  (r=0.92) is mechanically derived from the order's item prices. In a real pre-checkout prediction scenario, this would not be available. It must be **excluded or replaced** with a proxy (e.g., average category price from browsing history) during feature engineering.

### Multicollinearity
No feature pairs exceed |r| > 0.7 (excluding the target). VIF analysis will be performed in Module 4 after encoding.

![Correlation Heatmap](plots/09_correlation_heatmap.png)


## Summary of EDA Insights for Downstream Modules

1. **Cleaning (Module 3):** Filter canceled/unavailable orders; winsorize order value at R$519; translate Portuguese category names.
2. **Feature Engineering (Module 4):** Log-transform target; exclude  (leakage); encode categorical features (category, device, traffic, loyalty); create interaction features (income × loyalty).
3. **Modeling (Module 5):** Expect tree-based models to outperform linear due to non-linear category effects; synthetic features should add 5–15% incremental R².
4. **Business (Module 8):** Loyalty tier and traffic source are actionable levers — email campaigns to high-value segments, desktop-optimized checkout for high-AOV categories.
